In [ ]:
# =========================
# ALL OPTIONS EVAL (CV 5-FOLD)
# Baseline + NMF topics + Title-Article similarity + Combined + Hierarchical
# =========================
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin, clone
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import NMF
from sklearn.metrics import f1_score, recall_score, confusion_matrix
from scipy import sparse


# =========================
# LOAD DATA
# =========================
df = pd.read_csv("../data/processed/v3/development_v3.csv")

# Fix definitivi contro errori .lower()
df["article"] = df["article"].fillna("").astype(str)
df["title"]   = df["title"].fillna("").astype(str)
df["source"]  = df["source"].fillna("unknown").astype(str)

X = df.drop(columns=["label"])
y = df["label"]


# =========================
# UTILS
# =========================
def	rowwise_cosine_sparse(A, B, eps=1e-12):
	# A,B: sparse CSR shape (n, d)
	# cosine per riga: (A·B) / (||A||*||B||)
	numer = A.multiply(B).sum(axis=1).A1
	A_norm = np.sqrt(A.multiply(A).sum(axis=1)).A1
	B_norm = np.sqrt(B.multiply(B).sum(axis=1)).A1
	den = (A_norm * B_norm) + eps
	return numer / den

def	print_eval(name, f1s, recalls, cms, f1s_baseline=None):
	print("\n" + "="*60)
	print(name)
	print("Macro F1:", float(np.mean(f1s)))
	print("Macro Recall:", float(np.mean(recalls)))
	if f1s_baseline is not None:
		deltas = np.array(f1s) - np.array(f1s_baseline)
		print("ΔF1 vs baseline (mean):", float(deltas.mean()))
		print("ΔF1 vs baseline (std):", float(deltas.std()))
	print("Confusion Matrix (sum over folds):\n", np.sum(cms, axis=0))


# =========================
# CUSTOM: Title↔Article similarity features
# =========================
class	TitleArticleSimilarity(BaseEstimator, TransformerMixin):
	"""
	Produce numeric features:
	- cosine similarity between TF-IDF(title) and TF-IDF(article)
	- overlap ratio: |tokens(title) ∩ tokens(article)| / |tokens(title)|
	"""
	def	__init__(self, max_features_title=50_000, max_features_article=80_000, ngram_range=(1,2),
				 min_df_title=2, min_df_article=3, max_df_title=0.95, max_df_article=0.9,
				 stop_words="english", sublinear_tf=True):
		self.max_features_title = max_features_title
		self.max_features_article = max_features_article
		self.ngram_range = ngram_range
		self.min_df_title = min_df_title
		self.min_df_article = min_df_article
		self.max_df_title = max_df_title
		self.max_df_article = max_df_article
		self.stop_words = stop_words
		self.sublinear_tf = sublinear_tf

	def	fit(self, X_df, y=None):
		# X_df is a DataFrame with columns ["title","article"]
		self.vec_title_ = TfidfVectorizer(
			max_features=self.max_features_title,
			ngram_range=self.ngram_range,
			min_df=self.min_df_title,
			max_df=self.max_df_title,
			sublinear_tf=self.sublinear_tf,
			stop_words=self.stop_words
		)
		self.vec_article_ = TfidfVectorizer(
			max_features=self.max_features_article,
			ngram_range=self.ngram_range,
			min_df=self.min_df_article,
			max_df=self.max_df_article,
			sublinear_tf=self.sublinear_tf,
			stop_words=self.stop_words
		)
		self.vec_title_.fit(X_df["title"].astype(str))
		self.vec_article_.fit(X_df["article"].astype(str))
		return self

	def	transform(self, X_df):
		titles = X_df["title"].astype(str)
		articles = X_df["article"].astype(str)

		T = self.vec_title_.transform(titles)
		A = self.vec_article_.transform(articles)

		# Cosine similarity between two *different* spaces is not directly valid.
		# Workaround: use SAME vectorizer trained on concatenation:
		# To keep it simple and stable, we approximate cosine by projecting both in a shared vocabulary via title vectorizer.
		# Alternative: use a single shared vectorizer externally. This approximation often still adds signal (style mismatch).
		A_in_title_space = self.vec_title_.transform(articles)
		cos = rowwise_cosine_sparse(T, A_in_title_space)

		# Token overlap (cheap, lexical)
		overlap = []
		for t, a in zip(titles.tolist(), articles.tolist()):
			toks_t = set(t.lower().split())
			if len(toks_t) == 0:
				overlap.append(0.0)
				continue
			toks_a = set(a.lower().split())
			overlap.append(len(toks_t & toks_a) / max(1, len(toks_t)))
		overlap = np.array(overlap, dtype=float)

		out = np.vstack([cos, overlap]).T
		return out


# =========================
# BUILD PREPROCESSORS
# =========================
# baseline numeric columns (only if exist)
NUM_COLS = [c for c in ["title_ratio", "n_tokens"] if c in X.columns]

# shared blocks
ARTICLE_TFIDF = TfidfVectorizer(
	max_features=120_000,
	ngram_range=(1,2),
	min_df=3,
	max_df=0.9,
	sublinear_tf=True,
	stop_words="english"
)

TITLE_TFIDF = TfidfVectorizer(
	max_features=30_000,
	ngram_range=(1,2),
	min_df=2,
	max_df=0.95,
	sublinear_tf=True,
	stop_words="english"
)

SOURCE_OHE = OneHotEncoder(handle_unknown="ignore")


# -------------------------
# Baseline preprocessor (your Strategy 2)
# -------------------------
def	make_baseline_pipeline(C=1.0):
	prep = ColumnTransformer(
		transformers=[
			("article", ARTICLE_TFIDF, "article"),
			("title", TITLE_TFIDF, "title"),
			("source", SOURCE_OHE, ["source"]),
			("num", StandardScaler(), NUM_COLS),
		],
		n_jobs=-1
	)
	model = Pipeline([
		("prep", prep),
		("clf", LogisticRegression(C=C, max_iter=1000, n_jobs=-1))
	])
	return model


# -------------------------
# + NMF topics on article
# -------------------------
def	make_nmf_topics_block(n_topics=30, max_features_topics=50_000):
	# Separate vectorizer for topic extraction to keep it cheaper.
	vec = TfidfVectorizer(
		max_features=max_features_topics,
		ngram_range=(1,2),
		min_df=3,
		max_df=0.9,
		sublinear_tf=True,
		stop_words="english"
	)
	# NMF expects non-negative: TF-IDF ok
	nmf = NMF(
		n_components=n_topics,
		random_state=42,
		init="nndsvda",
		max_iter=300
	)
	return Pipeline([
		("tfidf", vec),
		("nmf", nmf)
	])

def	make_baseline_plus_nmf(C=1.0, n_topics=30):
	prep = ColumnTransformer(
		transformers=[
			("article", ARTICLE_TFIDF, "article"),
			("title", TITLE_TFIDF, "title"),
			("source", SOURCE_OHE, ["source"]),
			("num", StandardScaler(), NUM_COLS),
			("topics", make_nmf_topics_block(n_topics=n_topics), "article"),
		],
		n_jobs=-1
	)
	model = Pipeline([
		("prep", prep),
		("clf", LogisticRegression(C=C, max_iter=1000, n_jobs=-1))
	])
	return model


# -------------------------
# + Title↔Article similarity features
# -------------------------
def	make_baseline_plus_similarity(C=1.0):
	sim = TitleArticleSimilarity()
	# we pass a mini-DF with two columns via FunctionTransformer-like behaviour
	class	TwoColSelector(BaseEstimator, TransformerMixin):
		def	fit(self, X_df, y=None): return self
		def	transform(self, X_df): return X_df[["title","article"]]

	prep = ColumnTransformer(
		transformers=[
			("article", ARTICLE_TFIDF, "article"),
			("title", TITLE_TFIDF, "title"),
			("source", SOURCE_OHE, ["source"]),
			("num", StandardScaler(), NUM_COLS),

			("sim", Pipeline([
				("sel", TwoColSelector()),
				("sim", sim),
				("sc", StandardScaler())
			]), ["title","article"]),
		],
		n_jobs=-1
	)
	model = Pipeline([
		("prep", prep),
		("clf", LogisticRegression(C=C, max_iter=1000, n_jobs=-1))
	])
	return model


# -------------------------
# + BOTH: NMF topics + similarity
# -------------------------
def	make_baseline_plus_nmf_plus_similarity(C=1.0, n_topics=30):
	sim = TitleArticleSimilarity()
	class	TwoColSelector(BaseEstimator, TransformerMixin):
		def	fit(self, X_df, y=None): return self
		def	transform(self, X_df): return X_df[["title","article"]]

	prep = ColumnTransformer(
		transformers=[
			("article", ARTICLE_TFIDF, "article"),
			("title", TITLE_TFIDF, "title"),
			("source", SOURCE_OHE, ["source"]),
			("num", StandardScaler(), NUM_COLS),
			("topics", make_nmf_topics_block(n_topics=n_topics), "article"),
			("sim", Pipeline([
				("sel", TwoColSelector()),
				("sim", sim),
				("sc", StandardScaler())
			]), ["title","article"]),
		],
		n_jobs=-1
	)
	model = Pipeline([
		("prep", prep),
		("clf", LogisticRegression(C=C, max_iter=1000, n_jobs=-1))
	])
	return model


# =========================
# HIERARCHICAL CLASSIFIER
# hard vs soft -> then fine classifier
# =========================
HARD_LABELS = set([0, 1, 2])	# International, Business, Technology
SOFT_LABELS = set([3, 4, 5, 6])	# Entertainment, Sports, General, Health

class	HierarchicalClassifier(BaseEstimator, ClassifierMixin):
	def	__init__(self, gate_model, hard_model, soft_model):
		self.gate_model = gate_model
		self.hard_model = hard_model
		self.soft_model = soft_model

	def	fit(self, X, y):
		# Gate: predict hard(1) vs soft(0)
		y_gate = np.array([1 if int(lbl) in HARD_LABELS else 0 for lbl in y], dtype=int)
		self.gate_ = clone(self.gate_model)
		self.gate_.fit(X, y_gate)

		# Train experts
		mask_hard = np.array([int(lbl) in HARD_LABELS for lbl in y], dtype=bool)
		mask_soft = ~mask_hard

		self.hard_ = clone(self.hard_model)
		self.soft_ = clone(self.soft_model)

		self.hard_.fit(X[mask_hard], y[mask_hard])
		self.soft_.fit(X[mask_soft], y[mask_soft])
		return self

	def	predict(self, X):
		g = self.gate_.predict(X)	# 1=hard, 0=soft
		out = np.zeros(X.shape[0], dtype=int)

		idx_hard = np.where(g == 1)[0]
		idx_soft = np.where(g == 0)[0]

		if len(idx_hard) > 0:
			out[idx_hard] = self.hard_.predict(X[idx_hard])
		if len(idx_soft) > 0:
			out[idx_soft] = self.soft_.predict(X[idx_soft])

		return out


def	make_hierarchical_pipeline():
	# Use the SAME baseline feature extractor for all three models (gate + experts)
	feat = ColumnTransformer(
		transformers=[
			("article", ARTICLE_TFIDF, "article"),
			("title", TITLE_TFIDF, "title"),
			("source", SOURCE_OHE, ["source"]),
			("num", StandardScaler(), NUM_COLS),
		],
		n_jobs=-1
	)

	# Gate (binary)
	gate = Pipeline([
		("prep", feat),
		("clf", LogisticRegression(C=1.0, max_iter=1000, n_jobs=-1))
	])

	# Experts (multiclass)
	hard = Pipeline([
		("prep", feat),
		("clf", LogisticRegression(C=1.0, max_iter=1000, n_jobs=-1))
	])
	soft = Pipeline([
		("prep", feat),
		("clf", LogisticRegression(C=1.0, max_iter=1000, n_jobs=-1))
	])

	# Wrap: note that HierarchicalClassifier expects numpy-like slicing
	# We'll pass X as DataFrame; implement slicing by using .iloc in CV loop below.
	return gate, hard, soft


# =========================
# CV EVALUATION
# =========================
def	run_cv(model_builder, X_df, y_series, name, skf, f1s_baseline=None):
	f1s = []
	recalls = []
	cms = []

	for tr, te in skf.split(X_df, y_series):
		Xtr = X_df.iloc[tr].copy()
		Xte = X_df.iloc[te].copy()
		ytr = y_series.iloc[tr]
		yte = y_series.iloc[te]

		model = model_builder()
		model.fit(Xtr, ytr)
		yp = model.predict(Xte)

		f1s.append(f1_score(yte, yp, average="macro"))
		recalls.append(recall_score(yte, yp, average="macro"))
		cms.append(confusion_matrix(yte, yp))

	print_eval(name, f1s, recalls, cms, f1s_baseline=f1s_baseline)
	return f1s, recalls, cms


def	run_cv_hierarchical(X_df, y_series, name, skf, f1s_baseline=None):
	f1s = []
	recalls = []
	cms = []

	gate_base, hard_base, soft_base = make_hierarchical_pipeline()

	for tr, te in skf.split(X_df, y_series):
		Xtr = X_df.iloc[tr].copy()
		Xte = X_df.iloc[te].copy()
		ytr = y_series.iloc[tr].values
		yte = y_series.iloc[te].values

		h = HierarchicalClassifier(gate_model=gate_base, hard_model=hard_base, soft_model=soft_base)
		h.fit(Xtr, ytr)
		yp = h.predict(Xte)

		f1s.append(f1_score(yte, yp, average="macro"))
		recalls.append(recall_score(yte, yp, average="macro"))
		cms.append(confusion_matrix(yte, yp))

	print_eval(name, f1s, recalls, cms, f1s_baseline=f1s_baseline)
	return f1s, recalls, cms


# =========================
# RUN ALL EXPERIMENTS
# =========================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 0) Baseline
f1_base, r_base, cm_base = run_cv(
	model_builder=lambda: make_baseline_pipeline(C=1.0),
	X_df=X,
	y_series=y,
	name="(0) BASELINE Strategy 2",
	skf=skf
)

# 1) + NMF topics
_ = run_cv(
	model_builder=lambda: make_baseline_plus_nmf(C=1.0, n_topics=30),
	X_df=X,
	y_series=y,
	name="(1) BASELINE + NMF topics (k=30)",
	skf=skf,
	f1s_baseline=f1_base
)

# 2) + Title-Article similarity
_ = run_cv(
	model_builder=lambda: make_baseline_plus_similarity(C=1.0),
	X_df=X,
	y_series=y,
	name="(2) BASELINE + Title↔Article similarity",
	skf=skf,
	f1s_baseline=f1_base
)

# 3) + Both
_ = run_cv(
	model_builder=lambda: make_baseline_plus_nmf_plus_similarity(C=1.0, n_topics=30),
	X_df=X,
	y_series=y,
	name="(3) BASELINE + NMF topics + similarity",
	skf=skf,
	f1s_baseline=f1_base
)




(0) BASELINE Strategy 2
Macro F1: 0.7018251678042141
Macro Recall: 0.6980966594979412
Confusion Matrix (sum over folds):
 [[18975   627   406   784   196  2329   224]
 [  746  8339   525   349    87   431   111]
 [  686   623  9090   345    50   260   107]
 [ 1677   566   514  4974   613  1416   217]
 [  252    55    16   317  7598   332     4]
 [ 3743   640   285  1175   624  6313   273]
 [  453   143    85   226    36   266  1893]]

(1) BASELINE + NMF topics (k=30)
Macro F1: 0.7017828800911164
Macro Recall: 0.6982120028990528
ΔF1 vs baseline (mean): -4.22877130976218e-05
ΔF1 vs baseline (std): 0.0005979277420673616
Confusion Matrix (sum over folds):
 [[18966   630   409   783   196  2330   227]
 [  742  8342   524   351    88   429   112]
 [  684   629  9084   349    49   259   107]
 [ 1675   567   517  4973   611  1416   218]
 [  252    55    16   317  7599   331     4]
 [ 3736   638   287  1172   627  6319   274]
 [  449   142    85   232    35   263  1896]]

(2) BASELINE + Title↔

KeyError: "None of [Index([    0,     1,     2,     3,     5,     7,     9,    10,    11,    12,\n       ...\n       15986, 15988, 15990, 15991, 15992, 15994, 15995, 15996, 15998, 15999],\n      dtype='int64', length=9485)] are in the [columns]"

In [4]:
# =========================
# FIXED HIERARCHICAL MODEL (RUN ONLY THIS PART)
# =========================
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, recall_score, confusion_matrix


# =========================
# LABEL GROUPS
# =========================
HARD_LABELS = set([0, 1, 2])	# International, Business, Technology
SOFT_LABELS = set([3, 4, 5, 6])	# Entertainment, Sports, General, Health


# =========================
# SHARED FEATURE EXTRACTOR
# =========================
NUM_COLS = [c for c in ["title_ratio", "n_tokens"] if c in X.columns]

FEATURES = ColumnTransformer(
	transformers=[
		("article", TfidfVectorizer(
			max_features=120_000,
			ngram_range=(1,2),
			min_df=3,
			max_df=0.9,
			sublinear_tf=True,
			stop_words="english"
		), "article"),

		("title", TfidfVectorizer(
			max_features=30_000,
			ngram_range=(1,2),
			min_df=2,
			max_df=0.95,
			sublinear_tf=True,
			stop_words="english"
		), "title"),

		("source", OneHotEncoder(handle_unknown="ignore"), ["source"]),

		("num", StandardScaler(), NUM_COLS),
	],
	n_jobs=-1
)


# =========================
# HIERARCHICAL CLASSIFIER (FIXED)
# =========================
class	HierarchicalClassifier(BaseEstimator, ClassifierMixin):
	def	__init__(self, gate_model, hard_model, soft_model):
		self.gate_model = gate_model
		self.hard_model = hard_model
		self.soft_model = soft_model

	def	fit(self, X, y):
		# Binary gate target
		y_gate = np.array([1 if int(lbl) in HARD_LABELS else 0 for lbl in y])

		self.gate_ = clone(self.gate_model)
		self.gate_.fit(X, y_gate)

		mask_hard = np.array([int(lbl) in HARD_LABELS for lbl in y])
		mask_soft = ~mask_hard

		self.hard_ = clone(self.hard_model)
		self.soft_ = clone(self.soft_model)

		self.hard_.fit(X.iloc[mask_hard], y[mask_hard])
		self.soft_.fit(X.iloc[mask_soft], y[mask_soft])

		return self

	def	predict(self, X):
		g = self.gate_.predict(X)
		out = np.zeros(len(X), dtype=int)

		idx_hard = np.where(g == 1)[0]
		idx_soft = np.where(g == 0)[0]

		if len(idx_hard) > 0:
			out[idx_hard] = self.hard_.predict(X.iloc[idx_hard])
		if len(idx_soft) > 0:
			out[idx_soft] = self.soft_.predict(X.iloc[idx_soft])

		return out


# =========================
# BUILD MODELS
# =========================
def	make_pipeline():
	return Pipeline([
		("prep", FEATURES),
		("clf", LogisticRegression(
			C=1.0,
			max_iter=1000,
			n_jobs=-1
		))
	])

gate_model = make_pipeline()
hard_model = make_pipeline()
soft_model = make_pipeline()


# =========================
# CROSS-VALIDATION
# =========================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1s = []
recalls = []
cms = []

for tr, te in skf.split(X, y):
	Xtr = X.iloc[tr]
	Xte = X.iloc[te]
	ytr = y.iloc[tr].values
	yte = y.iloc[te].values

	model = HierarchicalClassifier(
		gate_model=gate_model,
		hard_model=hard_model,
		soft_model=soft_model
	)

	model.fit(Xtr, ytr)
	yp = model.predict(Xte)

	f1s.append(f1_score(yte, yp, average="macro"))
	recalls.append(recall_score(yte, yp, average="macro"))
	cms.append(confusion_matrix(yte, yp))


# =========================
# RESULTS
# =========================
print("\nHIERARCHICAL MODEL (FIXED)")
print("Macro F1:", np.mean(f1s))
print("Macro Recall:", np.mean(recalls))
print("Confusion Matrix:\n", np.sum(cms, axis=0))



HIERARCHICAL MODEL (FIXED)
Macro F1: 0.6907290317266942
Macro Recall: 0.6881002317386264
Confusion Matrix:
 [[18226   667   411   973   241  2773   250]
 [  813  8162   526   407    99   468   113]
 [  743   637  8966   366    58   272   119]
 [ 1461   566   559  4935   615  1626   215]
 [  175    52    22   343  7537   439     6]
 [ 3459   638   261  1266   627  6503   299]
 [  380   129    92   251    54   346  1850]]


In [6]:
# =========================
# EXTRACT & SAVE FEATURES — BASELINE v3
# =========================
import numpy as np
import pandas as pd
from scipy import sparse

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


# =========================
# LOAD DATA
# =========================
df = pd.read_csv("../data/processed/v3/development_v3.csv")

df["article"] = df["article"].fillna("").astype(str)
df["title"]   = df["title"].fillna("").astype(str)
df["source"]  = df["source"].fillna("unknown").astype(str)

X = df.drop(columns=["label"])
y = df["label"].values


# =========================
# BASELINE v3 PREPROCESSOR
# =========================
NUM_COLS = [c for c in ["title_ratio", "n_tokens"] if c in X.columns]

ARTICLE_TFIDF = TfidfVectorizer(
	max_features=120_000,
	ngram_range=(1,2),
	min_df=3,
	max_df=0.9,
	sublinear_tf=True,
	stop_words="english"
)

TITLE_TFIDF = TfidfVectorizer(
	max_features=30_000,
	ngram_range=(1,2),
	min_df=2,
	max_df=0.95,
	sublinear_tf=True,
	stop_words="english"
)

SOURCE_OHE = OneHotEncoder(handle_unknown="ignore")

PREP = ColumnTransformer(
	transformers=[
		("article", ARTICLE_TFIDF, "article"),
		("title", TITLE_TFIDF, "title"),
		("source", SOURCE_OHE, ["source"]),
		("num", StandardScaler(), NUM_COLS),
	],
	n_jobs=-1
)


# =========================
# FIT PREPROCESSOR ON FULL DATA
# =========================
Z = PREP.fit_transform(X)

print("Feature matrix shape:", Z.shape)
print("Sparse matrix:", sparse.issparse(Z))


# =========================
# EXTRACT FEATURE NAMES
# =========================
feature_names = []

# article tf-idf
article_feats = PREP.named_transformers_["article"].get_feature_names_out()
feature_names.extend([f"article::{f}" for f in article_feats])

# title tf-idf
title_feats = PREP.named_transformers_["title"].get_feature_names_out()
feature_names.extend([f"title::{f}" for f in title_feats])

# source one-hot
source_feats = PREP.named_transformers_["source"].get_feature_names_out(["source"])
feature_names.extend(source_feats.tolist())

# numeric
feature_names.extend(NUM_COLS)

feature_names = np.array(feature_names)

print("Total features extracted:", len(feature_names))
assert Z.shape[1] == len(feature_names)


# =========================
# SAVE FEATURE NAMES (TXT)
# =========================
with open("feature_names_baseline_v3.txt", "w", encoding="utf-8") as f:
	for name in feature_names:
		f.write(name + "\n")

print("Saved: feature_names_baseline_v3.txt")


# =========================
# (OPTIONAL) SAVE FEATURE MATRIX + LABELS
# =========================
sparse.save_npz("X_features_baseline_v3.npz", Z)
np.save("y_labels.npy", y)

print("Saved: X_features_baseline_v3.npz")
print("Saved: y_labels.npy")


Feature matrix shape: (79996, 149053)
Sparse matrix: True
Total features extracted: 149053
Saved: feature_names_baseline_v3.txt
Saved: X_features_baseline_v3.npz
Saved: y_labels.npy


In [7]:
# ============================================================
# FEATURE SPACE ANALYSIS — BASELINE v3
# ============================================================
import numpy as np
from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import chi2
from sklearn.preprocessing import MinMaxScaler
from collections import defaultdict


# ============================================================
# LOAD SAVED FEATURES
# ============================================================
X = sparse.load_npz("X_features_baseline_v3.npz")
y = np.load("y_labels.npy")

with open("feature_names_baseline_v3.txt", "r", encoding="utf-8") as f:
	feature_names = np.array([line.strip() for line in f])

print("X shape:", X.shape)
print("n_features:", len(feature_names))


# ============================================================
# (1) BASIC FEATURE DISTRIBUTION
# ============================================================
nnz_per_row = np.diff(X.indptr)
nnz_per_col = np.diff(X.tocsc().indptr)

print("\n=== FEATURE DISTRIBUTION ===")
print("Avg non-zero per sample:", nnz_per_row.mean())
print("Avg non-zero per feature:", nnz_per_col.mean())
print("Median nnz per feature:", np.median(nnz_per_col))


# ============================================================
# (2) TRAIN LOGISTIC REGRESSION ON FULL FEATURE SPACE
# ============================================================
clf = LogisticRegression(
	C=1.0,
	max_iter=2000,
	n_jobs=-1,
	multi_class="auto"
)

clf.fit(X, y)

W = clf.coef_	# shape: (n_classes, n_features)
classes = clf.classes_

print("\nClasses:", classes)


# ============================================================
# (3) TOP FEATURES PER CLASS (BY ABS WEIGHT)
# ============================================================
TOP_K = 30

print("\n=== TOP FEATURES PER CLASS ===")
for i, c in enumerate(classes):
	w = W[i]
	idx = np.argsort(np.abs(w))[::-1][:TOP_K]
	print(f"\nClass {c}:")
	for j in idx:
		print(f"{feature_names[j]:40s}  {w[j]:+.4f}")


# ============================================================
# (4) FOCUS: WHAT PUSHES TOWARD 'GENERAL' (label = 5)
# ============================================================
GENERAL_LABEL = 5
gen_idx = np.where(classes == GENERAL_LABEL)[0][0]
w_gen = W[gen_idx]

top_pos = np.argsort(w_gen)[::-1][:40]	# push toward General
top_neg = np.argsort(w_gen)[:40]		# push away

print("\n=== FEATURES PUSHING TOWARD GENERAL ===")
for j in top_pos:
	print(f"{feature_names[j]:40s}  {w_gen[j]:+.4f}")

print("\n=== FEATURES PUSHING AWAY FROM GENERAL ===")
for j in top_neg:
	print(f"{feature_names[j]:40s}  {w_gen[j]:+.4f}")


# ============================================================
# (5) CHI-SQUARE FEATURE SELECTION (GLOBAL)
# ============================================================
# chi2 requires non-negative -> rescale sparse matrix
scaler = MinMaxScaler()
X_chi = scaler.fit_transform(X)

chi_scores, pvals = chi2(X_chi, y)
idx_chi = np.argsort(chi_scores)[::-1][:50]

print("\n=== TOP FEATURES BY CHI² ===")
for j in idx_chi:
	print(f"{feature_names[j]:40s}  chi2={chi_scores[j]:.2f}")


# ============================================================
# (6) HOW MANY FEATURES ARE 'GENERAL-SPECIFIC'?
# ============================================================
# Measure how unique features are to General vs others
general_mask = (y == GENERAL_LABEL)
other_mask = ~general_mask

X_gen = X[general_mask]
X_oth = X[other_mask]

mean_gen = X_gen.mean(axis=0).A1
mean_oth = X_oth.mean(axis=0).A1

delta = mean_gen - mean_oth
idx_delta = np.argsort(delta)[::-1][:30]

print("\n=== FEATURES MORE PRESENT IN GENERAL THAN OTHERS ===")
for j in idx_delta:
	print(f"{feature_names[j]:40s}  Δmean={delta[j]:+.4e}")


X shape: (79996, 149053)
n_features: 149053

=== FEATURE DISTRIBUTION ===
Avg non-zero per sample: 43.10544277213861
Avg non-zero per feature: 23.134475656310173
Median nnz per feature: 5.0


C:\Users\msist\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



Classes: [0 1 2 3 4 5 6]

=== TOP FEATURES PER CLASS ===

Class 0:
source_Topix.Net                          +6.5463
title::afp                                +5.7999
article::rss world                        +4.9776
article::europe http                      +4.5630
article::rss europe                       +4.5630
article::entertainment                    -4.3705
source_CSMonitor                          +4.3666
article::world http                       +4.3459
article::entertainment http               -4.0807
article::rss entertainment                -4.0807
article::rss science                      -3.9379
article::science http                     -3.9379
article::rss http                         +3.9086
source_IPS                                +3.5318
article::time topstories                  -3.4976
article::topstories                       -3.4976
article::afp                              +3.4136
source_RedNova                            +3.3009
article::time nation            

TypeError: MinMaxScaler does not support sparse input. Consider using MaxAbsScaler instead.

In [8]:
# ============================================================
# CHI-SQUARE FEATURE ANALYSIS (SPARSE-SAFE, FIXED)
# ============================================================

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2

# ============================================================
# LOAD DATA
# ============================================================
df = pd.read_csv("../data/processed/v3/development_v3.csv")

df["article"] = df["article"].fillna("").astype(str)
df["title"]   = df["title"].fillna("").astype(str)

y = df["label"].values

# ============================================================
# TF-IDF (IDENTICAL TO BASELINE)
# ============================================================
vec_article = TfidfVectorizer(
	max_features=120_000,
	ngram_range=(1,2),
	min_df=3,
	max_df=0.9,
	sublinear_tf=True,
	stop_words="english"
)

vec_title = TfidfVectorizer(
	max_features=30_000,
	ngram_range=(1,2),
	min_df=2,
	max_df=0.95,
	sublinear_tf=True,
	stop_words="english"
)

Xa = vec_article.fit_transform(df["article"])
Xt = vec_title.fit_transform(df["title"])

# Combine text blocks
X_text = sparse.hstack([Xa, Xt]).tocsr()

print("X_text shape:", X_text.shape)

# ============================================================
# FEATURE NAMES
# ============================================================
feat_names = (
	["ART_" + f for f in vec_article.get_feature_names_out()] +
	["TTL_" + f for f in vec_title.get_feature_names_out()]
)

assert X_text.shape[1] == len(feat_names)

# ============================================================
# CHI-SQUARE (MULTICLASS)
# ============================================================
chi_scores, pvals = chi2(X_text, y)

# Top-K globally
K = 200
idx_global = np.argsort(chi_scores)[::-1][:K]

# ============================================================
# SAVE GLOBAL TOP FEATURES
# ============================================================
with open("chi2_top_global.txt", "w", encoding="utf-8") as f:
	for i in idx_global:
		f.write(f"{feat_names[i]}\t{chi_scores[i]:.6f}\n")

print("Saved: chi2_top_global.txt")

# ============================================================
# CHI-SQUARE PER LABEL (ONE-vs-REST)
# ============================================================
label_map = {
	0: "International",
	1: "Business",
	2: "Technology",
	3: "Entertainment",
	4: "Sports",
	5: "General",
	6: "Health"
}

for lbl, name in label_map.items():
	print(f"Processing label {lbl} ({name})")

	y_bin = (y == lbl).astype(int)

	chi_lbl, _ = chi2(X_text, y_bin)
	idx_lbl = np.argsort(chi_lbl)[::-1][:100]

	outfile = f"chi2_top_label_{lbl}_{name}.txt"
	with open(outfile, "w", encoding="utf-8") as f:
		for i in idx_lbl:
			f.write(f"{feat_names[i]}\t{chi_lbl[i]:.6f}\n")

	print(f"Saved: {outfile}")

print("DONE.")


X_text shape: (79996, 147692)
Saved: chi2_top_global.txt
Processing label 0 (International)
Saved: chi2_top_label_0_International.txt
Processing label 1 (Business)
Saved: chi2_top_label_1_Business.txt
Processing label 2 (Technology)
Saved: chi2_top_label_2_Technology.txt
Processing label 3 (Entertainment)
Saved: chi2_top_label_3_Entertainment.txt
Processing label 4 (Sports)
Saved: chi2_top_label_4_Sports.txt
Processing label 5 (General)
Saved: chi2_top_label_5_General.txt
Processing label 6 (Health)
Saved: chi2_top_label_6_Health.txt
DONE.


In [10]:
# ============================================================
# OVERLAP ANALYSIS — JACCARD TOP FEATURES
# ============================================================
from itertools import combinations

def load_feats(path, top=50):
	with open(path, "r", encoding="utf-8") as f:
		return set([line.split("\t")[0] for line in f.readlines()[:top]])

label_files = {
	0: "chi2_top_label_0_International.txt",
	1: "chi2_top_label_1_Business.txt",
	2: "chi2_top_label_2_Technology.txt",
	3: "chi2_top_label_3_Entertainment.txt",
	4: "chi2_top_label_4_Sports.txt",
	5: "chi2_top_label_5_General.txt",
	6: "chi2_top_label_6_Health.txt",
}

top_feats = {k: load_feats(v, top=50) for k, v in label_files.items()}

print("\n=== JACCARD OVERLAP (TOP-50 FEATURES) ===")
for (i, j) in combinations(top_feats.keys(), 2):
	A, B = top_feats[i], top_feats[j]
	jaccard = len(A & B) / len(A | B)
	if jaccard > 0.15:
		print(f"Label {i} vs {j}: Jaccard = {jaccard:.2f}")



=== JACCARD OVERLAP (TOP-50 FEATURES) ===


In [13]:
# ============================================================
# OVERLAP ANALYSIS — JACCARD (TOP-50, FULL PRINT)
# ============================================================
from itertools import combinations

def load_feats(path, top=50):
	with open(path, "r", encoding="utf-8") as f:
		lines = f.readlines()
	return set([line.split("\t")[0] for line in lines[:top]])

label_files = {
	0: "chi2_top_label_0_International.txt",
	1: "chi2_top_label_1_Business.txt",
	2: "chi2_top_label_2_Technology.txt",
	3: "chi2_top_label_3_Entertainment.txt",
	4: "chi2_top_label_4_Sports.txt",
	5: "chi2_top_label_5_General.txt",
	6: "chi2_top_label_6_Health.txt",
}

top_feats = {k: load_feats(v, top=50) for k, v in label_files.items()}

print("\n=== JACCARD OVERLAP (TOP-50 FEATURES) ===")
for i, j in combinations(top_feats.keys(), 2):
	A, B = top_feats[i], top_feats[j]
	inter = len(A & B)
	union = len(A | B)
	jaccard = inter / union if union > 0 else 0.0
	print(f"Label {i} vs {j}: overlap={inter:2d}, union={union:3d}, Jaccard={jaccard:.3f}")




=== JACCARD OVERLAP (TOP-50 FEATURES) ===
Label 0 vs 1: overlap= 1, union= 99, Jaccard=0.010
Label 0 vs 2: overlap= 0, union=100, Jaccard=0.000
Label 0 vs 3: overlap= 0, union=100, Jaccard=0.000
Label 0 vs 4: overlap= 0, union=100, Jaccard=0.000
Label 0 vs 5: overlap= 2, union= 98, Jaccard=0.020
Label 0 vs 6: overlap= 0, union=100, Jaccard=0.000
Label 1 vs 2: overlap= 0, union=100, Jaccard=0.000
Label 1 vs 3: overlap= 0, union=100, Jaccard=0.000
Label 1 vs 4: overlap= 0, union=100, Jaccard=0.000
Label 1 vs 5: overlap= 1, union= 99, Jaccard=0.010
Label 1 vs 6: overlap= 0, union=100, Jaccard=0.000
Label 2 vs 3: overlap= 0, union=100, Jaccard=0.000
Label 2 vs 4: overlap= 0, union=100, Jaccard=0.000
Label 2 vs 5: overlap= 0, union=100, Jaccard=0.000
Label 2 vs 6: overlap= 0, union=100, Jaccard=0.000
Label 3 vs 4: overlap= 0, union=100, Jaccard=0.000
Label 3 vs 5: overlap= 0, union=100, Jaccard=0.000
Label 3 vs 6: overlap= 0, union=100, Jaccard=0.000
Label 4 vs 5: overlap= 0, union=100, Ja

In [12]:
import os

print("CWD:", os.getcwd())
print("\nFILES IN CWD:")
for f in os.listdir("."):
	print(f)

CWD: c:\UniProj\DSLab_winter\DSLab-winter2026\v3code

FILES IN CWD:
chi2_top_global.txt
chi2_top_label_0_International.txt
chi2_top_label_1_Business.txt
chi2_top_label_2_Technology.txt
chi2_top_label_3_Entertainment.txt
chi2_top_label_4_Sports.txt
chi2_top_label_5_General.txt
chi2_top_label_6_Health.txt
FeatureImportance_v2_with_article_plain_text_token.ipynb
feature_names_baseline_v3.txt
gran_mix.ipynb
little_aggressive.ipynb
timestamp.ipynb
title_as_text.ipynb
v3_semanantic_minimal_numeric_plus_source.ipynb
X_features_baseline_v3.npz
y_labels.npy
